# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Learning-to-Rank (LTR) / Predictive Scoring

Why: Content teams have limited editorial bandwidth and cannot review thousands of declining pages individually. A binary classification model (needs_refresh: 0 or 1) is insufficient because over 2,400 pages show traffic drops, but editors can only tackle 50–100 per week. We need a predictive ranking/scoring model that orders pages by estimated ROI (traffic recovery potential) meaning which pages has higher chances of getting back on trending again so content managers can execute directly on the top of the queue.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

Target Definition: High-Opportunity Refresh Target (is_high_value_decay binary target or continuous opportunity_score_proxy).

Label Source: Defined composite rule based on observed performance signals.

Binary Target Variable: is_high_value_decay = 1 if trend_direction == 'down', impressions_90d >= 1000, and trend_pct <= -20%; else 0.Continuous Target (Proxy ROI):$$\text{Opportunity Score} = \text{impressions\_90d} \times \left\vert{} \frac{\text{trend\_pct}}{100} \right\vert{}$$Why a Proxy? We cannot observe true post-refresh traffic gains until an editor actually rewrites a page. Therefore, historical traffic drop severity combined with active search demand (impressions_90d) serves as our defensible ground-truth proxy for opportunity.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Defensible Metric: Precision@50 (and NDCG@50)Target Benchmark:

$\mathbf{\text{Precision@50} \ge 85\%}$

Defense:Global metrics like Overall Accuracy or ROC-AUC are misleading because predicting dead pages correctly inflates overall model performance.Content managers review only the top 50–100 recommendations in their weekly batch. If 85%+ of the top 50 flagged pages are genuine, high-value refresh candidates with significant search impressions and severe traffic decay, the system builds strong operational trust with editorial teams.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [3]:
import pandas as pd
import numpy as np

# Load starter data
df = pd.read_excel('capstone_data.xlsx')

# Clean numeric trend column
df['trend_pct_num'] = pd.to_numeric(df['trend_pct'], errors='coerce').fillna(0)

# Define Unit of Analysis: 1 row = 1 unique content page (content_id)
# Define binary target flag
df['is_high_value_decay'] = (
    (df['trend_direction'] == 'down') &
    (df['impressions_90d'] >= 1000) &
    (df['trend_pct_num'] <= -20)
).astype(int)

# Continuous baseline opportunity proxy
df['trend_drop_severity'] = np.where(df['trend_pct_num'] < 0, np.abs(df['trend_pct_num']) / 100.0, 0)
df['opportunity_score_proxy'] = df['impressions_90d'] * df['trend_drop_severity']

# Select slice to show unit of analysis
unit_df = df[[
    'content_id',
    'impressions_90d',
    'clicks_90d',
    'trend_pct_num',
    'content_age_days',
    'engagement_rate',
    'is_high_value_decay',
    'opportunity_score_proxy'
]].copy()

print(f"Unit of Analysis Dataset Shape: {unit_df.shape}")
print("\nTarget Label Distribution (is_high_value_decay):")
print(unit_df['is_high_value_decay'].value_counts(normalize=True))

# Show top rows ranked by baseline opportunity
unit_df.sort_values(by='opportunity_score_proxy', ascending=False).head(10)

Unit of Analysis Dataset Shape: (4467, 8)

Target Label Distribution (is_high_value_decay):
is_high_value_decay
0    0.73987
1    0.26013
Name: proportion, dtype: float64


,content_id,impressions_90d,clicks_90d,trend_pct_num,content_age_days,engagement_rate,is_high_value_decay,opportunity_score_proxy
2041,content_551fe371f51b,115789,47,-84.1,224,4.17,1,97378.549
3343,content_54baba704595,130617,8,-54.8,286,1.42,1,71578.116
1448,content_62ed76850efc,167858,1272,-41.6,224,2.02,1,69828.928
578,content_b51e2e4d22ff,91795,33,-74.6,141,4.00,1,68479.070
1548,content_e9c6e67086f6,126441,263,-45.3,124,0.97,1,57277.773
2382,content_65114d89496d,72631,12,-74.8,482,0.00,1,54327.988
2825,content_7ec1abc04dec,80957,415,-59.1,126,2.62,1,47845.587
482,content_39881853ef0c,112434,10,-42.0,97,3.45,1,47222.280
2529,content_824a11467353,75376,112,-58.2,480,3.97,1,43868.832
939,content_302dff6caa63,113892,167,-38.1,417,0.00,1,43392.852


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

Why IF-STATEMENTS Fail:

Multi-Signal Trade-offs: A fixed rule like if impressions > 1000 and trend < -20% treats signals in isolation. It cannot evaluate non-linear interactions—such as a page with moderate impression drops but severe CTR degradation vs. a page with high drop severity caused purely by seasonal demand shifts.

Arbitrary Hard Boundaries: Simple threshold rules create strict edge cases (e.g., a page with 999 impressions is completely ignored while one with 1,000 is flagged). ML models learn smooth weightings across content age, ranking positions, scroll rates, and trend decay slopes.

Scalability & Explainability: ML models allow us to rank candidates holistically and extract dynamic feature importances / reason codes (e.g., declining CTR on position 1–3 vs. high impressions on stale page) without maintaining hundreds of fragile if-else branches.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.